# 02 — Human review & judge alignment

Select 10 production traces, collect human labels (live Review App or shortcut), and align the `escalation_correctness` judge to the human pattern. Independent of `01_eval_and_promote` (different 10 cases).

**Prerequisite:** `00_production_traffic_and_monitoring` has run AND production monitoring has scored these traces — the LLM-judge `escalation_correctness` assessment must already be on the traces (from monitoring) so `align()` can pair it with the human labels. No eval dataset is needed.

## Install dependencies

Same as 00_production_traffic_and_monitoring — needed when running 01 as a standalone job (MLflow 3 GenAI APIs). Skip if the cluster already has them.

In [ ]:
%pip install -U mlflow databricks-sdk openai python-dotenv dspy databricks-agents

In [ ]:
dbutils.library.restartPython()

In [ ]:
import sys, json
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer
from src import config
from src.scorers import escalation_correctness

mlflow.set_experiment(config.MLFLOW_EXPERIMENT_NAME)
EXPERIMENT_ID = mlflow.get_experiment_by_name(config.MLFLOW_EXPERIMENT_NAME).experiment_id
tickets = [r.asDict(recursive=True) for r in spark.table(config.TICKETS_TABLE).orderBy('ticket_id').collect()]
tickets_by_id = {t['ticket_id']: t for t in tickets}
print(f'{len(tickets)} tickets loaded')

## Step 1 — Select 10 traces for review

Select **7 ambiguous + 3 should-escalate** traces from production traffic, restricted to tickets we have human labels for (so the shortcut path can label all 10). Different cases from `01`.

In [ ]:
# Production traffic = traces logged by 00_production_traffic_and_monitoring (eval runs created later carry a run_id, which we filter out below).
all_traces = mlflow.search_traces(experiment_ids=[EXPERIMENT_ID], max_results=500, return_type='list')
prod_traces = [t for t in all_traces if not getattr(t.info, 'run_id', None)]

def trace_to_ticket_id(trace):
    inputs = trace.data.spans[0].inputs if trace.data.spans else {}
    if isinstance(inputs, dict):
        ticket = inputs.get('ticket') or inputs
        if isinstance(ticket, dict):
            return ticket.get('ticket_id')
    return None

labeled_ids = {r['ticket_id'] for r in spark.table(config.HUMAN_LABELS_TABLE).select('ticket_id').collect()}
selected = []
for t in prod_traces:
    tid = trace_to_ticket_id(t)
    if not tid or tid not in labeled_ids:
        continue
    selected.append({'trace': t, 'ticket_id': tid, 'bucket': tickets_by_id[tid]['_truth']['difficulty_bucket']})

ambiguous = [r for r in selected if r['bucket'] == 'ambiguous'][:7]
should_escalate = [r for r in selected if r['bucket'] == 'should_escalate'][:3]
review_queue = ambiguous + should_escalate
print(f'Review queue: {len(review_queue)} traces ({len(ambiguous)} ambiguous + {len(should_escalate)} should-escalate)')

## Step 3 — Human review (choose a path)

Set `USE_LIVE_REVIEW` in the next cell:
- **`True` (default)** — create an MLflow labeling session, attach the 10 traces, and print a Review App link; reviewers label each trace and the answers are written back as HUMAN feedback.
- **`False`** — load pre-recorded labels and log them as HUMAN feedback (instant fallback).

Both paths end with an `escalation_correctness` feedback assessment (`source_type=HUMAN`) on each trace. The label-schema name must be `escalation_correctness` (matching the judge name) so `align()` can pair human vs. judge scores.

Live path (default): create the session, label all 10 in the Review App, then continue to Step 4. (Steps 4+ need the labels to exist, so finish labeling before running them.)

In [ ]:
# === Choose the review path ===
USE_LIVE_REVIEW = True  # True = create a live labeling session (Review App); False = log pre-recorded labels

LABEL_SCHEMA_NAME = 'escalation_correctness'  # MUST match the judge name so align() can pair scores

if USE_LIVE_REVIEW:
    # --- Live path: create a labeling session + Review App link for the 10 traces ---
    import mlflow.genai.labeling as labeling
    import mlflow.genai.label_schemas as label_schemas

    # Reviewer answers "was the agent's escalate decision correct?" -> correct / incorrect.
    # Schema name MUST equal the judge name (escalation_correctness) for align() to pair scores.
    label_schemas.create_label_schema(
        name=LABEL_SCHEMA_NAME,
        type='feedback',
        title='Was the escalation decision correct?',
        input=label_schemas.InputCategorical(options=['correct', 'incorrect']),
        instruction=(
            "Read the ticket and the agent's output (category, escalate, draft). "
            "Decide whether the agent's escalate / do-not-escalate decision was correct, "
            "and add a brief rationale."
        ),
        enable_comment=True,
        overwrite=True,
    )

    current_user = spark.sql('SELECT current_user()').first()[0]
    session = labeling.create_labeling_session(
        name='support_triage_escalation_review',
        assigned_users=[current_user],
        label_schemas=[LABEL_SCHEMA_NAME],
    )
    session.add_traces([entry['trace'] for entry in review_queue])

    print(f'Labeling session created with {len(review_queue)} traces, assigned to {current_user}.')
    print(f'\n>>> Open the Review App to label all {len(review_queue)} traces:\n{session.url}\n')
    print('When every trace is labeled, continue to Step 4.')
else:
    # --- Shortcut: log pre-recorded human labels as HUMAN feedback (instant) ---
    labels_by_id = {l['ticket_id']: l for l in
                    [r.asDict(recursive=True) for r in spark.table(config.HUMAN_LABELS_TABLE).collect()]}
    for entry in review_queue:
        label = labels_by_id.get(entry['ticket_id'])
        if not label:
            continue
        agent_escalate = entry['trace'].data.spans[0].outputs.get('escalate', None)
        mlflow.log_feedback(
            trace_id=entry['trace'].info.trace_id,
            name=LABEL_SCHEMA_NAME,
            value='correct' if label['correct_should_escalate'] == agent_escalate else 'incorrect',
            rationale=label['reviewer_notes'],
            source=AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id='pre_recorded_reviewer'),
        )
    print(f'Logged {len(review_queue)} human-source feedback entries (shortcut)')

## Step 4 — Judge-human disagreement

Pull the reviewed traces. Each carries two `escalation_correctness` assessments — one `LLM_JUDGE`, one `HUMAN`. Count where they disagree.

Disagreement concentrates in the ambiguous bucket; that gap is what the alignment step closes.

In [ ]:
reviewed_trace_ids = {r['trace'].info.trace_id for r in review_queue}
# Re-fetch traces so the HUMAN feedback logged in Step 3 is attached
# (the trace objects loaded in Step 1 are stale and predate that feedback).
# Re-fetch run-less production traces (no run_id filter — traces are logged run-less now)
fresh_traces = [
    t for t in mlflow.search_traces(experiment_ids=[EXPERIMENT_ID], max_results=500, return_type='list')
    if not getattr(t.info, 'run_id', None)
]
reviewed_traces = [t for t in fresh_traces if t.info.trace_id in reviewed_trace_ids]

def _assessments(trace, name):
    # Assessments may be exposed via search_assessments() or trace.info.assessments
    if hasattr(trace, 'search_assessments'):
        try:
            return trace.search_assessments(name=name) or []
        except Exception:
            pass
    info = getattr(trace, 'info', None)
    alist = getattr(info, 'assessments', None) or getattr(trace, 'assessments', None) or []
    return [a for a in alist if getattr(a, 'name', getattr(a, 'assessment_name', None)) == name]

def _value(a):
    fb = getattr(a, 'feedback', None)
    if fb is not None and hasattr(fb, 'value'):
        return fb.value
    return getattr(a, 'value', None)

def _src_type(a):
    st = getattr(getattr(a, 'source', None), 'source_type', None)
    return getattr(st, 'value', st)  # enum -> its value, else the string itself

def get_assessment(trace, name, source_type):
    for a in _assessments(trace, name):
        if _src_type(a) == source_type:
            return _value(a)
    return None

agreements, disagreements = 0, 0
for t in reviewed_traces:
    judge_val = get_assessment(t, 'escalation_correctness', 'LLM_JUDGE')
    human_val = get_assessment(t, 'escalation_correctness', 'HUMAN')
    if judge_val is None or human_val is None:
        continue
    if judge_val == human_val:
        agreements += 1
    else:
        disagreements += 1

total = agreements + disagreements
if total:
    print(f'Judge-human agreement on reviewed traces: {agreements}/{total} ({100*agreements/total:.0f}%)')
    print(f'Disagreement: {disagreements}/{total} ({100*disagreements/total:.0f}%)')
else:
    print('No paired assessments yet — run Step 3 first, then re-run scorers if needed.')

## Step 5 — Align the judge

Pass the reviewed traces (now carrying both LLM and HUMAN assessments) to `SIMBAAlignmentOptimizer`. It refines the judge's instructions to better match the human labels and returns an `aligned_judge` (same name, refined rubric).

In [ ]:
valid_traces = [
    t for t in reviewed_traces
    if get_assessment(t, 'escalation_correctness', 'HUMAN') is not None
    and get_assessment(t, 'escalation_correctness', 'LLM_JUDGE') is not None
]
print(f'{len(valid_traces)} traces have both human and judge assessments (minimum 10 needed)')

if len(valid_traces) >= 10:
    optimizer = SIMBAAlignmentOptimizer(model=f'databricks:/{config.JUDGE_MODEL}')
    # Judge.align signature is align(traces, optimizer) -> use keywords to be safe
    aligned_judge = escalation_correctness.align(traces=valid_traces, optimizer=optimizer)
    # Persist the aligned judge under a new name (best-effort; not required for the re-score below)
    try:
        from mlflow.genai.judges import make_judge
        make_judge(
            name='escalation_correctness_aligned',
            instructions=aligned_judge.instructions,
            model=f'databricks:/{config.JUDGE_MODEL}',
        ).register(experiment_id=EXPERIMENT_ID)
        print('Aligned judge registered as escalation_correctness_aligned')
    except Exception as e:
        print(f'Aligned judge ready; registration skipped ({type(e).__name__}: {e})')
else:
    aligned_judge = escalation_correctness
    print('Not enough valid traces — falling back to initial judge')